In [ ]:
import srt
from datetime import timedelta


def json_to_word_srt(data):
    subtitles = []
    index = 1

    events = data.get("events", [])

    for i, event in enumerate(events):

        if "segs" not in event:
            continue

        event_start = event.get("tStartMs", 0)

        # Determine event end time
        if "dDurationMs" in event:
            event_end = event_start + event["dDurationMs"]
        else:
            # Find next event with a start time
            event_end = None
            for j in range(i + 1, len(events)):
                if "tStartMs" in events[j]:
                    event_end = events[j]["tStartMs"]
                    break

            if event_end is None:
                event_end = event_start + 1000  # fallback 1 second

        # Extract valid words
        words = []
        for seg in event["segs"]:
            text = seg.get("utf8", "").strip()

            if not text or text == "\n":
                continue

            start_ms = event_start + seg.get("tOffsetMs", 0)
            words.append((text, start_ms))

        if not words:
            continue

        # Create one subtitle per word
        for k, (word, start_ms) in enumerate(words):

            if k < len(words) - 1:
                end_ms = words[k + 1][1]
            else:
                end_ms = event_end

            # Prevent invalid timestamps
            if end_ms <= start_ms:
                end_ms = start_ms + 50

            subtitles.append(
                srt.Subtitle(
                    index=index,
                    start=timedelta(milliseconds=start_ms),
                    end=timedelta(milliseconds=end_ms),
                    content=word,
                )
            )
            index += 1

    return srt.compose(subtitles)



In [ ]:
import pytubefix as pytube
from pytubefix.innertube import InnerTube
# pytube.YouTube.from_id("sWehw-D5whM").chapters
p = pytube.YouTube.from_id("a7V1j5WFCl8")


In [ ]:
p.captions

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
ytt_api = YouTubeTranscriptApi()
transcript_list = ytt_api.list('wk62YFS3gqc')


In [ ]:
import subprocess
import os

input_video = "input.mp4"
subtitle_file = "subtitle.srt"
output_video = "output_soft.mp4"

# Check files
if not os.path.exists(input_video):
    raise FileNotFoundError(f"Video not found: {input_video}")

if not os.path.exists(subtitle_file):
    raise FileNotFoundError(f"Subtitle file not found: {subtitle_file}")

# FFmpeg command
command = [
    "ffmpeg",
    "-y",
    "-i", input_video,
    "-i", subtitle_file,
    "-c:v", "copy",          # Video without re-encoding
    "-c:a", "copy",          # Audio without re-encoding
    "-c:s", "mov_text",      # MP4 subtitle format
    "-metadata:s:s:0", "language=eng",
    output_video
]

try:
    subprocess.run(command, check=True)
    print(f"Done! Soft subtitles added to: {output_video}")
except subprocess.CalledProcessError as e:
    print("FFmpeg failed!")
    print(e)

In [ ]:
subtitle_content = """1
00:00:00,000 --> 00:00:04,000
Namaste! Yeh ek dummy subtitle test hai.

2
00:00:04,000 --> 00:00:08,000
Is line ka istemal subtitle embedding check ke liye hai.

3
00:00:08,000 --> 00:00:12,000
Agar yeh text dikh raha hai, sab sahi chal raha hai.

4
00:00:12,000 --> 00:00:16,000
Python aur FFmpeg ka demo subtitle.

5
00:00:16,000 --> 00:00:20,000
Dummy subtitle test complete. Dhanyavaad!
"""

with open("subtitle.srt", "w", encoding="utf-8") as f:
    f.write(subtitle_content)

print("subtitle.srt created successfully!")